# Digital Economy Intelligence — Equipo 4 (Ciencia)
**Actividad 2 · Parte I** — Diagnóstico comparativo de economía digital: **México, Estados Unidos, Alemania, China y Japón**.

**Cómo reproducir:** (1) `pip install -r requirements.txt`; (2) descargar el CSV de ITU indicado en la sección 1.2 a `data/raw/`; (3) *Run All*. El notebook consulta la API oficial del Banco Mundial, guarda las respuestas originales en `data/raw/` (sin modificarlas) y genera `data/processed/digital_economy_clean.csv`, `source_log.csv` y `data_dictionary.csv`. Después: `streamlit run dashboard/app.py`.

**Regla de oro:** no hay datos sintéticos ni completados a mano. Un valor no disponible queda vacío (NaN) y nunca se confunde con cero.

## 0. Configuración

In [1]:
import sys, re, warnings, unicodedata
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True); PROC.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))

from dei_core import *          # configuración de indicadores, DRS, clustering, textos
from dei_plots import *         # figuras Plotly

START_YEAR, END_YEAR = 2015, date.today().year   # ventana de consulta a la API
MIN_YEAR = 2018                                  # un dato anterior a este año se considera NO disponible (demasiado antiguo)
TODAY = date.today().isoformat()
pd.options.display.float_format = "{:,.3f}".format
print("Raíz del proyecto:", ROOT)

Raíz del proyecto: /home/rama/Desktop/UPY/digital_economy/u1/Team04_DigitalEconomy


## 1. Adquisición e integración de datos

### 1.1 Indicadores seleccionados (exactamente 8)
Distribución exigida: 2 infraestructura/conectividad · 1 acceso/uso · 1 calidad/asequibilidad · 2 actividad económica digital · 2 capacidad tecnológica/innovación. Como el equipo es de **Ciencia**, se refuerza la dimensión de capacidad tecnológica (I7, I8) con la selección y con los pesos del DRS.

In [2]:
sel = pd.DataFrame([{k: i[k] for k in ["id", "short", "dimension", "unit", "code", "organismo", "plataforma"]} for i in INDICATORS])
display(sel)
assert len(INDICATORS) == 8
print(sel.dimension.value_counts().to_string())

,id,short,dimension,unit,code,organismo,plataforma
0,I1,Banda ancha fija,Infraestructura y conectividad,por 100 habitantes,IT.NET.BBND.P2,ITU (World Telecommunication/ICT Indicators Da...,World Bank Indicators API
1,I2,Servidores seguros,Infraestructura y conectividad,por millón de habitantes,IT.NET.SECR.P6,"Netcraft, compilado por el Banco Mundial",World Bank Indicators API
2,I3,Uso de Internet,Acceso y uso,% de la población,IT.NET.USER.ZS,ITU (World Telecommunication/ICT Indicators Da...,World Bank Indicators API
3,I4,Costo banda ancha (% INB pc),Calidad y asequibilidad,% del INB per cápita,ITU DataHub (descarga CSV),ITU,ITU DataHub (descarga manual de CSV oficial)
4,I5,Exportaciones servicios TIC,Actividad económica digital,% de exportaciones de servicios,BX.GSR.CCIS.ZS,"FMI (Balanza de Pagos), compilado por el Banco...",World Bank Indicators API
5,I6,Exportaciones bienes TIC,Actividad económica digital,% de exportaciones de bienes,TX.VAL.ICTG.ZS.UN,"Naciones Unidas (UN Comtrade), compilado por e...",World Bank Indicators API
6,I7,Gasto en I+D (% PIB),Capacidad tecnológica e innovación,% del PIB,GB.XPD.RSDV.GD.ZS,"UNESCO Institute for Statistics, compilado por...",World Bank Indicators API
7,I8,Patentes residentes /millón,Capacidad tecnológica e innovación,solicitudes por millón de habitantes,IP.PAT.RESD / SP.POP.TOTL,WIPO (patentes) y Banco Mundial (población),World Bank Indicators API (o WIPO IP Statistic...


dimension
Infraestructura y conectividad        2
Actividad económica digital           2
Capacidad tecnológica e innovación    2
Acceso y uso                          1
Calidad y asequibilidad               1


### 1.2 Procedimiento de descarga (reproducible)
* **I1, I2, I3, I5, I6, I7, I8:** API v2 del Banco Mundial (`https://api.worldbank.org/v2/country/MEX;USA;DEU;CHN;JPN/indicator/<CÓDIGO>?format=json`). Cada respuesta original se guarda tal cual en `data/raw/wb_<CÓDIGO>.json`. Varias series son compiladas por otros organismos (ITU, UNESCO-UIS, FMI, UN Comtrade, WIPO); el Source Log registra el organismo original y la plataforma de acceso.
* **I4 (asequibilidad):** no está en la API del Banco Mundial. Descargar desde **ITU DataHub** (https://datahub.itu.int/) el indicador *Fixed-broadband basket as a percentage of GNI per capita* para las cinco economías (todos los años) en CSV y guardarlo **sin editar** como `data/raw/itu_fixed_broadband_basket.csv`.
* **Opcional (fuente directa WIPO):** si se descarga de *WIPO IP Statistics Data Center* la serie de solicitudes de patente de residentes y se guarda como `data/raw/wipo_patents_residents.csv`, el notebook la usa en lugar de la serie WIPO servida por el Banco Mundial.

In [3]:
WB_API = "https://api.worldbank.org/v2"
ISO_LIST = ";".join(ISO3)

def wb_get(url, params, raw_name):
    """Consulta la API y guarda la respuesta ORIGINAL sin modificar en data/raw/."""
    last = None
    for _ in range(3):
        try:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            (RAW / raw_name).write_text(r.text, encoding="utf-8")
            return r.json()
        except Exception as e:      # reintento simple
            last = e
    raise RuntimeError(f"No se pudo consultar {url}: {last}")

def wb_series(code):
    payload = wb_get(f"{WB_API}/country/{ISO_LIST}/indicator/{code}",
                     dict(format="json", date=f"{START_YEAR}:{END_YEAR}", per_page=1000), f"wb_{code}.json")
    if not isinstance(payload, list) or len(payload) < 2 or not payload[1]:
        raise ValueError(f"La API no devolvió datos para {code}: {payload}")
    if payload[0].get("pages", 1) > 1:
        raise ValueError("Respuesta paginada: aumentar per_page")
    df = pd.DataFrame([{"iso3": d["countryiso3code"], "year": int(d["date"]), "value": d["value"]} for d in payload[1]])
    df["value"] = pd.to_numeric(df["value"])      # null -> NaN (dato no disponible, NO cero)
    return df[df.iso3.isin(ISO3)]

def wb_meta(code):
    m = wb_get(f"{WB_API}/indicator/{code}", dict(format="json"), f"wb_meta_{code}.json")[1][0]
    return m["name"], (m.get("sourceNote") or "").strip()

WB_CODES = {i["col"]: i["code"] for i in INDICATORS if i["plataforma"].startswith("World Bank") and i["id"] != "I8"}
series, official = {}, {}
for col, cod in WB_CODES.items():
    series[col] = wb_series(cod)
    official[col] = wb_meta(cod)
    print(f"{cod:22s} filas={len(series[col]):3d}  con dato={series[col].value.notna().sum():3d}")

IT.NET.BBND.P2         filas= 55  con dato= 49


IT.NET.SECR.P6         filas= 55  con dato= 50


IT.NET.USER.ZS         filas= 55  con dato= 51


BX.GSR.CCIS.ZS         filas= 55  con dato= 55


TX.VAL.ICTG.ZS.UN      filas= 55  con dato= 50


GB.XPD.RSDV.GD.ZS      filas= 55  con dato= 46


**Lector de CSV oficiales descargados manualmente (ITU / WIPO).** Detecta automáticamente columnas de país, año y valor (formato largo o ancho) y conserva solo las cinco economías. Si el archivo trae varios indicadores, filtra por palabra clave y falla con un mensaje claro si sigue habiendo ambigüedad.

In [4]:
def _norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower().strip()
    return re.sub(r"\s+", " ", s)

NAME2ISO = {"mexico": "MEX", "united states": "USA", "united states of america": "USA", "usa": "USA", "us": "USA",
            "estados unidos": "USA", "germany": "DEU", "alemania": "DEU", "china": "CHN",
            "china, people's republic of": "CHN", "people's republic of china": "CHN",
            "japan": "JPN", "japon": "JPN"}

def load_generic_csv(path, indicator_filter=None):
    df = pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")
    key = {c: re.sub(r"[^a-z0-9]", "", str(c).lower()) for c in df.columns}
    def find(cands):
        for cand in cands:
            for c, k in key.items():
                if k == cand:
                    return c
    iso_c = find(["entityiso", "iso3", "isocode", "iso", "countrycode", "economyiso", "code"])
    name_c = find(["entityname", "country", "countryname", "economy", "economyname", "geographicarea", "origin",
                   "originname", "entity", "area", "reportingeconomy", "office"])
    year_c = find(["datayear", "year", "timeperiod", "time", "period", "obstime", "date"])
    val_c = find(["datavalue", "value", "obsvalue", "obs", "observationvalue", "figure"])
    ind_c = find(["indicator", "indicatorname", "seriesname", "series", "indicatorcode", "seriescode"])
    if year_c is None or val_c is None:            # formato ancho: años como columnas
        ycols = [c for c in df.columns if re.fullmatch(r"\d{4}", str(c).strip())]
        if not ycols:
            raise ValueError(f"No se reconoce el formato de {path.name}. Columnas: {list(df.columns)}")
        ids = [c for c in [iso_c, name_c, ind_c] if c]
        df = df.melt(id_vars=ids, value_vars=ycols, var_name="year", value_name="value")
        year_c, val_c = "year", "value"
    if ind_c and indicator_filter and df[ind_c].nunique() > 1:
        df = df[df[ind_c].astype(str).str.contains(indicator_filter, case=False, regex=True)]
    if ind_c and df[ind_c].nunique() > 1:
        raise ValueError(f"{path.name} contiene varios indicadores; deja solo uno. Encontrados: {df[ind_c].unique()[:10]}")
    def to_iso(row):
        if iso_c and str(row[iso_c]).strip().upper() in ISO3:
            return str(row[iso_c]).strip().upper()
        n = _norm(row[name_c]) if name_c else ""
        if n in NAME2ISO: return NAME2ISO[n]
        return "CHN" if n.startswith("china") and not any(x in n for x in ["hong", "macao", "taiwan"]) else None
    df["iso3"] = df.apply(to_iso, axis=1)
    out = pd.DataFrame({"iso3": df["iso3"],
                        "year": pd.to_numeric(df[year_c].astype(str).str.extract(r"(\d{4})")[0], errors="coerce"),
                        "value": pd.to_numeric(df[val_c].astype(str).str.replace(",", ""), errors="coerce")})
    out = out[out.iso3.isin(ISO3) & out.year.notna()]
    out["year"] = out.year.astype(int)
    dup = out.duplicated(["iso3", "year"], keep=False)
    if dup.any():
        warnings.warn(f"{path.name}: {dup.sum()} filas duplicadas iso3-año; se conserva la última. Revisar.")
        out = out.drop_duplicates(["iso3", "year"], keep="last")
    return out.reset_index(drop=True)

ITU_FILE = RAW / "itu_fixed_broadband_basket.csv"
if not ITU_FILE.exists():
    raise FileNotFoundError("Falta data/raw/itu_fixed_broadband_basket.csv — descárgalo de ITU DataHub (ver sección 1.2).")
series["broadband_basket"] = load_generic_csv(ITU_FILE, indicator_filter="basket")
print("ITU basket: filas", len(series["broadband_basket"]), "| países:", sorted(series["broadband_basket"].iso3.unique()))

ITU basket: filas 5 | países: ['CHN', 'DEU', 'JPN', 'MEX', 'USA']


In [5]:
# I8: patentes de residentes por millón de habitantes
WIPO_FILE = RAW / "wipo_patents_residents.csv"
if WIPO_FILE.exists():
    patents = load_generic_csv(WIPO_FILE); PAT_PLATFORM = "WIPO IP Statistics Data Center (CSV en data/raw)"
else:
    patents = wb_series("IP.PAT.RESD"); wb_meta("IP.PAT.RESD"); PAT_PLATFORM = "World Bank Indicators API (serie de WIPO)"
pop = wb_series("SP.POP.TOTL")
m = patents.merge(pop, on=["iso3", "year"], suffixes=("_pat", "_pop"))
m["value"] = m["value_pat"] / m["value_pop"] * 1_000_000     # transformación documentada
series["patents_per_million"] = m[["iso3", "year", "value"]]
official["patents_per_million"] = (INDICATORS[7]["official_name"], INDICATORS[7]["definition"])
print("Fuente de patentes:", PAT_PLATFORM)

Fuente de patentes: World Bank Indicators API (serie de WIPO)


## 2. Comparabilidad temporal
**Criterio:** para cada indicador se toma el **año más reciente en el que las cinco economías tienen dato** (año común, ≥ `MIN_YEAR`). Si no existe un año común, se toma el último dato de cada país (≥ `MIN_YEAR`) y **se reporta** que los años difieren. Un dato anterior a `MIN_YEAR`, o inexistente, se deja como **no disponible** (NaN); nunca se rellena ni se imputa.

**Riesgos de comparar años distintos:** (i) shocks coyunturales (pandemia, ciclos de inversión) afectan a unos años y no a otros; (ii) los indicadores de adopción crecen rápido, así que un rezago de 2–3 años puede equivaler a una diferencia real grande; (iii) los ratios (% de exportaciones, % del PIB) cambian con el denominador; (iv) revisiones metodológicas de los organismos. La comparación es engañosa sobre todo cuando el rezago es mayor a ~2 años en indicadores de rápida evolución (uso de Internet, banda ancha) o cuando un país queda rezagado justo en el indicador donde se le quiere evaluar.

In [6]:
def select_observations(long_df, min_year=MIN_YEAR):
    wide = long_df.pivot_table(index="year", columns="iso3", values="value", aggfunc="last").reindex(columns=ISO3)
    wide = wide[wide.index >= min_year]
    complete = wide.dropna(how="any")
    if len(complete):
        y = int(complete.index.max())
        return pd.DataFrame({"value": wide.loc[y], "year": y}), f"Año común más reciente con las 5 economías: {y}"
    rows = {}
    for iso in ISO3:
        s = wide[iso].dropna()
        rows[iso] = (int(s.index[-1]), float(s.iloc[-1])) if len(s) else (np.nan, np.nan)
    df = pd.DataFrame(rows, index=["year", "value"]).T
    return df[["value", "year"]], "No hay año común: último dato disponible por país (años distintos; ver columna *_year)"

picked, methods = {}, {}
for ind in INDICATORS:
    picked[ind["col"]], methods[ind["col"]] = select_observations(series[ind["col"]])

values = pd.DataFrame({c: picked[c]["value"] for c in COLS}).loc[ISO3]
years = pd.DataFrame({c: picked[c]["year"] for c in COLS}).loc[ISO3]

resumen = pd.DataFrame({"criterio": methods, "año_mín": years.min(), "año_máx": years.max()})
resumen["dispersión_años"] = resumen["año_máx"] - resumen["año_mín"]
display(resumen)
display(years.rename(index=COUNTRIES))
if (resumen["dispersión_años"] > 0).any():
    display(Markdown("⚠️ **Hay indicadores con años distintos entre países**: interpretar con cautela (ver riesgos arriba)."))

,criterio,año_mín,año_máx,dispersión_años
fixed_broadband,Año común más reciente con las 5 economías: 2023,2023,2023,0
secure_servers,Año común más reciente con las 5 economías: 2024,2024,2024,0
internet_users,Año común más reciente con las 5 economías: 2024,2024,2024,0
broadband_basket,Año común más reciente con las 5 economías: 2025,2025,2025,0
ict_services_exports,Año común más reciente con las 5 economías: 2025,2025,2025,0
ict_goods_exports,Año común más reciente con las 5 economías: 2024,2024,2024,0
rd_expenditure,Año común más reciente con las 5 economías: 2023,2023,2023,0
patents_per_million,Año común más reciente con las 5 economías: 2021,2021,2021,0


,fixed_broadband,secure_servers,internet_users,broadband_basket,ict_services_exports,ict_goods_exports,rd_expenditure,patents_per_million
iso3,,,,,,,,
México,2023,2024,2024,2025,2025,2024,2023,2021
Estados Unidos,2023,2024,2024,2025,2025,2024,2023,2021
Alemania,2023,2024,2024,2025,2025,2024,2023,2021
China,2023,2024,2024,2025,2025,2024,2023,2021
Japón,2023,2024,2024,2025,2025,2024,2023,2021


## 3. Exploración y preparación
Se revisan faltantes, tipos, escalas, valores extremos e inconsistencias. **No se imputa nada automáticamente.**

In [7]:
raw_table = values.rename(index=COUNTRIES)
display(raw_table)
print("Tipos:\n", values.dtypes.to_string())
faltantes = values.isna().sum().rename("faltantes")
display(faltantes.to_frame().T)

# Inconsistencias de rango
problemas = []
for col in ["internet_users", "ict_services_exports", "ict_goods_exports"]:
    bad = values.index[(values[col] < 0) | (values[col] > 100)].tolist()
    if bad: problemas.append(f"{col}: fuera de 0–100 en {bad}")
for col in COLS:
    bad = values.index[values[col] < 0].tolist()
    if bad: problemas.append(f"{col}: valores negativos en {bad}")
print("Inconsistencias de rango:", problemas or "ninguna")

# Ceros vs. no disponibles
print("Valores exactamente cero:", int((values == 0).sum().sum()), "| No disponibles (NaN):", int(values.isna().sum().sum()))

# Valores extremos (n=5: se usa una regla simple, > 3 veces la mediana del grupo)
ext = []
for col in COLS:
    med = values[col].median()
    for iso, v in values[col].items():
        if pd.notna(v) and med and v > 3 * med:
            ext.append(f"{COUNTRIES[iso]} en {META[col]['short']} ({v:,.2f} vs mediana {med:,.2f})")
print("Valores extremos (>3x mediana):", ext or "ninguno")
display(values.describe().T)

,fixed_broadband,secure_servers,internet_users,broadband_basket,ict_services_exports,ict_goods_exports,rd_expenditure,patents_per_million
iso3,,,,,,,,
México,20.753,531.656,83.122,1.666,5.089,12.450,0.267,8.751
Estados Unidos,38.094,"196,616.101",94.694,0.760,8.298,9.050,3.447,789.653
Alemania,45.381,"152,113.844",93.500,0.740,11.473,4.550,3.154,478.652
China,44.729,"1,413.321",92.000,0.379,19.135,21.320,2.577,"1,010.114"
Japón,38.631,"32,928.758",85.542,1.007,5.989,7.640,3.441,"1,769.965"


Tipos:
 fixed_broadband         float64
secure_servers          float64
internet_users          float64
broadband_basket        float64
ict_services_exports    float64
ict_goods_exports       float64
rd_expenditure          float64
patents_per_million     float64


,fixed_broadband,secure_servers,internet_users,broadband_basket,ict_services_exports,ict_goods_exports,rd_expenditure,patents_per_million
faltantes,0,0,0,0,0,0,0,0


Inconsistencias de rango: ninguna
Valores exactamente cero: 0 | No disponibles (NaN): 0
Valores extremos (>3x mediana): ['Estados Unidos en Servidores seguros (196,616.10 vs mediana 32,928.76)', 'Alemania en Servidores seguros (152,113.84 vs mediana 32,928.76)']


,count,mean,std,min,25%,50%,75%,max
fixed_broadband,5.000,37.517,9.956,20.753,38.094,38.631,44.729,45.381
secure_servers,5.000,"76,720.736","91,450.450",531.656,"1,413.321","32,928.758","152,113.844","196,616.101"
internet_users,5.000,89.772,5.128,83.122,85.542,92.000,93.500,94.694
broadband_basket,5.000,0.910,0.478,0.379,0.740,0.760,1.007,1.666
ict_services_exports,5.000,9.997,5.673,5.089,5.989,8.298,11.473,19.135
ict_goods_exports,5.000,11.002,6.428,4.550,7.640,9.050,12.450,21.320
rd_expenditure,5.000,2.577,1.339,0.267,2.577,3.154,3.441,3.447
patents_per_million,5.000,811.427,654.556,8.751,478.652,789.653,"1,010.114","1,769.965"


In [8]:
decisiones = [
    "Se usan solo las 5 economías asignadas (MEX, USA, DEU, CHN, JPN) y los 8 indicadores definidos; se descartan agregados regionales y otros países.",
    f"Ventana de consulta {START_YEAR}–{END_YEAR}; datos anteriores a {MIN_YEAR} se consideran no disponibles.",
    "Año de observación: año común más reciente por indicador; si no existe, último dato por país y se reporta (sección 2).",
    "Valores faltantes: se conservan como NaN; NO se imputan ni se sustituyen por cero.",
    "I8 = patentes de residentes / población × 1,000,000, con el mismo año para numerador y denominador.",
    "I4 (costo de la canasta) tiene sentido inverso: se invierte al normalizar (menor costo = mejor).",
    "Los valores extremos NO se eliminan: son datos reales (p. ej., China en patentes); su efecto se discute en el DRS y en la sensibilidad.",
    "Ningún archivo de data/raw/ fue modificado.",
]
display(Markdown("**Decisiones de preparación**\n\n" + "\n".join(f"{n+1}. {d}" for n, d in enumerate(decisiones))))

**Decisiones de preparación**

1. Se usan solo las 5 economías asignadas (MEX, USA, DEU, CHN, JPN) y los 8 indicadores definidos; se descartan agregados regionales y otros países.
2. Ventana de consulta 2015–2026; datos anteriores a 2018 se consideran no disponibles.
3. Año de observación: año común más reciente por indicador; si no existe, último dato por país y se reporta (sección 2).
4. Valores faltantes: se conservan como NaN; NO se imputan ni se sustituyen por cero.
5. I8 = patentes de residentes / población × 1,000,000, con el mismo año para numerador y denominador.
6. I4 (costo de la canasta) tiene sentido inverso: se invierte al normalizar (menor costo = mejor).
7. Los valores extremos NO se eliminan: son datos reales (p. ej., China en patentes); su efecto se discute en el DRS y en la sensibilidad.
8. Ningún archivo de data/raw/ fue modificado.

### 3.1 Dataset integrado y documentación (Source Log y Data Dictionary)

In [9]:
clean = pd.DataFrame({"iso3": ISO3, "country": [COUNTRIES[c] for c in ISO3]})
for c in COLS:
    clean[c] = values[c].values
    clean[f"{c}_year"] = years[c].astype("Int64").values
assert list(clean.iso3) == ISO3 and len(clean) == 5
clean.to_csv(PROC / "digital_economy_clean.csv", index=False, encoding="utf-8")

yrs = {c: years[c].to_dict() for c in COLS}
log = build_source_log(years=yrs, methods=methods, official=official, retrieved=TODAY)
log.loc[log.id == "I8", "plataforma_acceso"] = PAT_PLATFORM
log.to_csv(ROOT / "source_log.csv", index=False, encoding="utf-8")
build_data_dictionary().to_csv(ROOT / "data_dictionary.csv", index=False, encoding="utf-8")
display(log[["id", "nombre_oficial", "unidad", "organismo_original", "plataforma_acceso", "codigo_indicador", "anio_observacion_por_pais"]])
display(clean)

,id,nombre_oficial,unidad,organismo_original,plataforma_acceso,codigo_indicador,anio_observacion_por_pais
0,I1,Fixed broadband subscriptions (per 100 people),por 100 habitantes,ITU (World Telecommunication/ICT Indicators Da...,World Bank Indicators API,IT.NET.BBND.P2,MEX:2023; USA:2023; DEU:2023; CHN:2023; JPN:2023
1,I2,Secure Internet servers (per 1 million people),por millón de habitantes,"Netcraft, compilado por el Banco Mundial",World Bank Indicators API,IT.NET.SECR.P6,MEX:2024; USA:2024; DEU:2024; CHN:2024; JPN:2024
2,I3,Individuals using the Internet (% of population),% de la población,ITU (World Telecommunication/ICT Indicators Da...,World Bank Indicators API,IT.NET.USER.ZS,MEX:2024; USA:2024; DEU:2024; CHN:2024; JPN:2024
3,I4,Fixed-broadband basket as a percentage of GNI ...,% del INB per cápita,ITU,ITU DataHub (descarga manual de CSV oficial),ITU DataHub (descarga CSV),MEX:2025; USA:2025; DEU:2025; CHN:2025; JPN:2025
4,I5,"ICT service exports (% of service exports, BoP)",% de exportaciones de servicios,"FMI (Balanza de Pagos), compilado por el Banco...",World Bank Indicators API,BX.GSR.CCIS.ZS,MEX:2025; USA:2025; DEU:2025; CHN:2025; JPN:2025
5,I6,ICT goods exports (% of total goods exports),% de exportaciones de bienes,"Naciones Unidas (UN Comtrade), compilado por e...",World Bank Indicators API,TX.VAL.ICTG.ZS.UN,MEX:2024; USA:2024; DEU:2024; CHN:2024; JPN:2024
6,I7,Research and development expenditure (% of GDP),% del PIB,"UNESCO Institute for Statistics, compilado por...",World Bank Indicators API,GB.XPD.RSDV.GD.ZS,MEX:2023; USA:2023; DEU:2023; CHN:2023; JPN:2023
7,I8,"Patent applications, residents (IP.PAT.RESD) p...",solicitudes por millón de habitantes,WIPO (patentes) y Banco Mundial (población),World Bank Indicators API (serie de WIPO),IP.PAT.RESD / SP.POP.TOTL,MEX:2021; USA:2021; DEU:2021; CHN:2021; JPN:2021


,iso3,country,fixed_broadband,fixed_broadband_year,secure_servers,secure_servers_year,internet_users,internet_users_year,broadband_basket,broadband_basket_year,ict_services_exports,ict_services_exports_year,ict_goods_exports,ict_goods_exports_year,rd_expenditure,rd_expenditure_year,patents_per_million,patents_per_million_year
0,MEX,México,20.753,2023,531.656,2024,83.122,2024,1.666,2025,5.089,2025,12.450,2024,0.267,2023,8.751,2021
1,USA,Estados Unidos,38.094,2023,"196,616.101",2024,94.694,2024,0.760,2025,8.298,2025,9.050,2024,3.447,2023,789.653,2021
2,DEU,Alemania,45.381,2023,"152,113.844",2024,93.500,2024,0.740,2025,11.473,2025,4.550,2024,3.154,2023,478.652,2021
3,CHN,China,44.729,2023,"1,413.321",2024,92.000,2024,0.379,2025,19.135,2025,21.320,2024,2.577,2023,"1,010.114",2021
4,JPN,Japón,38.631,2023,"32,928.758",2024,85.542,2024,1.007,2025,5.989,2025,7.640,2024,3.441,2023,"1,769.965",2021


## 4. Fundamentación conceptual

### A. Sector TIC, economía digital y economía digitalizada
* **Sector TIC:** conjunto de actividades cuya función principal es **producir o distribuir** tecnologías de la información y la comunicación: fabricación de equipos electrónicos y de cómputo, telecomunicaciones, desarrollo de software y servicios informáticos. Se delimita por *industria* (p. ej., divisiones 26, 61, 62 y 63 de la CIIU Rev. 4).
* **Economía digital:** es más amplia. Incluye el sector TIC y además las actividades cuyo modelo de negocio **depende** de insumos digitales, datos y plataformas: comercio electrónico, plataformas, servicios digitalmente entregados, tecnología financiera, etc. Se delimita por el *tipo de actividad*.
* **Economía digitalizada:** es el **uso** de tecnologías digitales por empresas, gobiernos y hogares en el resto de los sectores (agricultura de precisión, manufactura conectada, banca en línea). No es un sector: es un proceso de adopción transversal.

**¿Cuál describe mejor nuestros indicadores?** La **economía digital en sentido amplio, con enfoque de preparación**. I1–I4 miden la base de infraestructura, adopción y asequibilidad (condiciones para que exista economía digital y para digitalizar otros sectores); I5–I6 miden actividad TIC comercializada (mezcla de sector TIC y servicios digitalmente entregables); I7–I8 miden capacidad de generar tecnología. No es solo sector TIC porque la mayoría de los indicadores no miden producción TIC; tampoco es economía digitalizada porque no observamos adopción sectorial (p. ej., uso de nube o de comercio electrónico por empresas).

### B. Crecimiento de la economía digital
Tres indicadores, tres dimensiones distintas:
* **I3 – Uso de Internet (% población):** *adopción*. Indica qué parte de la sociedad participa en el mundo digital (demanda potencial, inclusión), no si el país produce algo.
* **I5 – Exportaciones de servicios TIC (% de exportaciones de servicios):** *captura de valor y competitividad externa*. Muestra si la economía vende capacidades digitales al mundo.
* **I8 – Patentes de residentes por millón de habitantes:** *capacidad tecnológica e innovación*. Muestra generación local de conocimiento protegible, es decir, dependencia o autonomía tecnológica.

**¿Por qué un solo indicador no basta?** El desarrollo digital es multidimensional y las dimensiones pueden divergir: un país puede tener alta adopción pero depender de tecnología importada; puede exportar muchos bienes TIC porque ensambla (valor agregado bajo) sin innovar; o puede tener mucha infraestructura poco utilizada por su costo. Un indicador solo captura una cara, es sensible a su definición y a su denominador, y puede ocultar desigualdades internas (urbano/rural, empresas grandes/pequeñas).

In [10]:
i7 = "rd_expenditure"
v = values[i7]; otros = v.drop("MEX")
rk = int(v.rank(ascending=False, method="min")["MEX"])
display(Markdown(f"""
### C. Indicador, evidencia e interpretación — I7 Gasto en I+D (% del PIB)
**Dato observado.** México destinó **{v['MEX']:.2f}% del PIB** a I+D (año {int(years.loc['MEX', i7])}); las otras economías: {", ".join(f"{COUNTRIES[c]} {x:.2f}%" for c, x in otros.items())}. México ocupa el lugar **{rk} de 5**.

**Interpretación.** El indicador mide el *esfuerzo* (insumo) de la economía por generar conocimiento: qué proporción de su producción se reinvierte en investigación, tanto pública como privada. Un valor {"inferior" if v['MEX'] < otros.mean() else "superior"} al promedio de las otras cuatro economías ({otros.mean():.2f}%) indica una intensidad de I+D {"menor" if v['MEX'] < otros.mean() else "mayor"} en relación con el tamaño de su economía.

**Conclusión sustentada.** Con este dato puede afirmarse que la **intensidad de inversión en I+D de México es {"menor" if v['MEX'] < otros.mean() else "mayor"}** que la del promedio de este grupo de referencia, medida como proporción del PIB.

**Lo que NO puede concluirse solo con este dato:** que México *sea menos innovador* (no vemos resultados: patentes, publicaciones, productos), que el gasto sea *eficiente* o de *calidad*, cuánto lo financia el sector privado frente al público, ni que México *no esté preparado* para IA. Tampoco explica *por qué* el nivel es ese (políticas, estructura productiva, tamaño de empresas).
"""))


### C. Indicador, evidencia e interpretación — I7 Gasto en I+D (% del PIB)
**Dato observado.** México destinó **0.27% del PIB** a I+D (año 2023); las otras economías: Estados Unidos 3.45%, Alemania 3.15%, China 2.58%, Japón 3.44%. México ocupa el lugar **5 de 5**.

**Interpretación.** El indicador mide el *esfuerzo* (insumo) de la economía por generar conocimiento: qué proporción de su producción se reinvierte en investigación, tanto pública como privada. Un valor inferior al promedio de las otras cuatro economías (3.15%) indica una intensidad de I+D menor en relación con el tamaño de su economía.

**Conclusión sustentada.** Con este dato puede afirmarse que la **intensidad de inversión en I+D de México es menor** que la del promedio de este grupo de referencia, medida como proporción del PIB.

**Lo que NO puede concluirse solo con este dato:** que México *sea menos innovador* (no vemos resultados: patentes, publicaciones, productos), que el gasto sea *eficiente* o de *calidad*, cuánto lo financia el sector privado frente al público, ni que México *no esté preparado* para IA. Tampoco explica *por qué* el nivel es ese (políticas, estructura productiva, tamaño de empresas).


### D. Comercio digital
* **Transacción de comercio electrónico:** compra o venta de bienes o servicios realizada **por redes de computadoras**, con métodos diseñados específicamente para recibir o hacer pedidos (sitio web, app, EDI). Lo definitorio es cómo se **hace el pedido**; la entrega puede ser física. *Ejemplo:* comprar unos tenis en un marketplace y recibirlos por paquetería.
* **Servicio digitalmente entregado:** servicio que se **presta y entrega remotamente** por redes de TIC (la entrega es digital), haya sido o no contratado en línea. *Ejemplo:* una empresa mexicana de desarrollo de software que presta consultoría y entrega código a un cliente en EE. UU. por videollamada y repositorios.
* **Relación:** son conceptos que se traslapan pero no son iguales. Comprar un libro electrónico o una suscripción de streaming es ambas cosas; comprar zapatos en línea es comercio electrónico pero no servicio digitalmente entregado; un contrato de consultoría cerrado por correo y entregado por internet es servicio digitalmente entregado, aunque no se haya contratado en una plataforma de comercio electrónico.

## 5. Digital Readiness Score (DRS)

**Fórmula:** `DRS = w1·I1 + w2·I2 + … + w8·I8`, con `Σ w = 1`, donde cada `Ij` es el indicador normalizado (0–1) y el resultado se expresa en 0–100.

**Normalización — min-max entre las cinco economías:** `Ij = (x − mín) / (máx − mín)`. Se eligió porque es transparente, conserva las proporciones relativas y es fácil de explicar. *Costo:* el índice es **relativo** (el peor país de cada indicador obtiene 0 y el mejor 1) y sensible a valores extremos; por eso se hace un análisis de sensibilidad.

**Sentido de las variables:** I1–I3 y I5–I8, mayor es mejor. **I4 (costo de la canasta de banda ancha, % del INB per cápita) es inverso**: `Ij = 1 − (x − mín)/(máx − mín)`.

**Pesos (justificación):** infraestructura 20% (10+10): condición habilitante; acceso/uso 15%; asequibilidad 15% (determina si la infraestructura se usa); actividad económica digital 20% (10+10; ponderación baja para I6 porque exportar bienes TIC puede reflejar ensamblaje y no captura de valor); capacidad tecnológica e innovación **30%** (15+15) por ser el sector asignado a este equipo (Ciencia).

**Datos faltantes:** no se imputan; si un país no tuviera un indicador, sus pesos se **reponderan** entre los indicadores disponibles y se reporta la cobertura.

In [11]:
w = weights()
assert abs(sum(w.values()) - 1) < 1e-9, "Los pesos deben sumar 1"
pesos = pd.DataFrame({"indicador": [f'{META[c]["id"]} {META[c]["short"]}' for c in COLS], "dimensión": [META[c]["dimension"] for c in COLS],
                      "sentido": ["↑ mayor es mejor" if META[c]["direction"] > 0 else "↓ mayor es peor (se invierte)" for c in COLS],
                      "peso": [w[c] for c in COLS]})
display(pesos); print("Suma de pesos =", sum(w.values()))

N, contrib, drs, coverage = compute_drs(values)
dim = dimension_scores(N)
display(Markdown("**Indicadores normalizados (0–100)**")); display((N * 100).rename(index=COUNTRIES))
display(Markdown("**Resultados DRS**"))
res_drs = pd.DataFrame({"DRS": drs, "lugar": drs.rank(ascending=False, method="min").astype(int), "cobertura_de_pesos": coverage}).sort_values("DRS", ascending=False)
display(res_drs.rename(index=COUNTRIES))

,indicador,dimensión,sentido,peso
0,I1 Banda ancha fija,Infraestructura y conectividad,↑ mayor es mejor,0.100
1,I2 Servidores seguros,Infraestructura y conectividad,↑ mayor es mejor,0.100
2,I3 Uso de Internet,Acceso y uso,↑ mayor es mejor,0.150
3,I4 Costo banda ancha (% INB pc),Calidad y asequibilidad,↓ mayor es peor (se invierte),0.150
4,I5 Exportaciones servicios TIC,Actividad económica digital,↑ mayor es mejor,0.100
5,I6 Exportaciones bienes TIC,Actividad económica digital,↑ mayor es mejor,0.100
6,I7 Gasto en I+D (% PIB),Capacidad tecnológica e innovación,↑ mayor es mejor,0.150
7,I8 Patentes residentes /millón,Capacidad tecnológica e innovación,↑ mayor es mejor,0.150


Suma de pesos = 1.0


**Indicadores normalizados (0–100)**

,fixed_broadband,secure_servers,internet_users,broadband_basket,ict_services_exports,ict_goods_exports,rd_expenditure,patents_per_million
iso3,,,,,,,,
México,0.000,0.000,0.000,0.000,0.000,47.108,0.000,0.000
Estados Unidos,70.413,100.000,100.000,70.382,22.848,26.834,100.000,44.339
Alemania,100.000,77.305,89.683,71.912,45.453,0.000,90.777,26.681
China,97.354,0.450,76.720,100.000,100.000,100.000,72.643,56.856
Japón,72.595,16.522,20.907,51.157,6.410,18.426,99.814,100.000


**Resultados DRS**

,DRS,lugar,cobertura_de_pesos
iso3,,,
China,75.713,1,1.000
Estados Unidos,69.218,2,1.000
Alemania,64.134,3,1.000
Japón,52.177,4,1.000
México,4.711,5,1.000


In [12]:
# Sensibilidad: (a) pesos iguales, (b) sin I6, (c) z-score en lugar de min-max
eq = {c: 1 / 8 for c in COLS}
no6 = {**{c: META[c]["weight"] for c in COLS}, "ict_goods_exports": 0.0}
drs_eq = compute_drs(values, eq)[2]
drs_no6 = compute_drs(values, no6)[2]
Z = (values - values.mean()) / values.std(ddof=0)
for i in INDICATORS:
    if i["direction"] < 0: Z[i["col"]] = -Z[i["col"]]
drs_z = (Z.fillna(0) * pd.Series(weights())).sum(axis=1)
sens = pd.DataFrame({"DRS base": drs, "pesos iguales": drs_eq, "sin I6": drs_no6}).round(1)
ranks = pd.DataFrame({"base": drs.rank(ascending=False), "pesos iguales": drs_eq.rank(ascending=False),
                      "sin I6": drs_no6.rank(ascending=False), "z-score": drs_z.rank(ascending=False)}).astype(int)
display(sens.rename(index=COUNTRIES)); display(ranks.rename(index=COUNTRIES))
estable = (ranks.nunique(axis=1) == 1)
display(Markdown("**Robustez del ranking:** " + ("el orden de las cinco economías es idéntico en todos los escenarios." if estable.all()
        else "cambia la posición de " + ", ".join(COUNTRIES[c] for c in estable[~estable].index) + " según el escenario; conviene leer el ranking con cautela.")))

,DRS base,pesos iguales,sin I6
iso3,,,
México,4.700,5.900,0.000
Estados Unidos,69.200,66.900,73.900
Alemania,64.100,62.700,71.300
China,75.700,75.500,73.000
Japón,52.200,48.200,55.900


,base,pesos iguales,sin I6,z-score
iso3,,,,
México,5,5,5,5
Estados Unidos,2,2,1,2
Alemania,3,3,3,3
China,1,1,2,1
Japón,4,4,4,4


**Robustez del ranking:** cambia la posición de Estados Unidos, China según el escenario; conviene leer el ranking con cautela.

## 6. Visualización de datos

### Visualización 1 — ¿Cuál es la posición relativa de las cinco economías y qué indicadores explican su DRS?

In [13]:
fig1 = fig_ranking(contrib, drs); fig1.show()
display(Markdown(f"""**¿Qué observo?** {describe_ranking(drs, contrib)}

**¿Qué significa?** El DRS ordena a las economías *dentro de este grupo*: mide preparación relativa, no un nivel absoluto. Las barras apiladas muestran de qué indicadores proviene cada puntaje.

**¿Qué no puedo concluir?** Que una economía con menor DRS esté "atrasada" en términos absolutos, ni que la diferencia de puntos sea proporcional a la diferencia real de preparación (por la normalización min-max y los pesos elegidos)."""))

**¿Qué observo?** Orden por DRS: 1. China (75.7); 2. Estados Unidos (69.2); 3. Alemania (64.1); 4. Japón (52.2); 5. México (4.7). México ocupa el lugar 5 de 5; la distancia entre el primero y el último es de 71.0 puntos y el indicador que más aporta al DRS de México es «Exportaciones bienes TIC» (4.7 puntos).

**¿Qué significa?** El DRS ordena a las economías *dentro de este grupo*: mide preparación relativa, no un nivel absoluto. Las barras apiladas muestran de qué indicadores proviene cada puntaje.

**¿Qué no puedo concluir?** Que una economía con menor DRS esté "atrasada" en términos absolutos, ni que la diferencia de puntos sea proporcional a la diferencia real de preparación (por la normalización min-max y los pesos elegidos).

### Visualización 2 — ¿Cuáles son las fortalezas y brechas de México frente a las otras cuatro economías?

In [14]:
fig2 = fig_gap(N); fig2.show()
display(Markdown(f"""**¿Qué observo?** {describe_gap(N)}

**¿Qué significa?** Las barras a la derecha son indicadores donde México supera al promedio del grupo; las de la izquierda, donde queda por debajo. El tamaño resume la distancia en la escala relativa 0–100.

**¿Qué no puedo concluir?** Que la brecha se deba a una causa específica (política pública, estructura productiva, tamaño de mercado) ni que cerrarla sea el mejor uso de recursos; el promedio de cuatro países no es una meta normativa."""))

**¿Qué observo?** Frente al promedio normalizado de las otras cuatro economías, México está por encima en: Exportaciones bienes TIC (+11); y por debajo en: Gasto en I+D (% PIB) (-91), Banda ancha fija (-85), Costo banda ancha (% INB pc) (-73), Uso de Internet (-72), Patentes residentes /millón (-57), Servidores seguros (-49) y Exportaciones servicios TIC (-44) (diferencias en puntos de la escala 0-100).

**¿Qué significa?** Las barras a la derecha son indicadores donde México supera al promedio del grupo; las de la izquierda, donde queda por debajo. El tamaño resume la distancia en la escala relativa 0–100.

**¿Qué no puedo concluir?** Que la brecha se deba a una causa específica (política pública, estructura productiva, tamaño de mercado) ni que cerrarla sea el mejor uso de recursos; el promedio de cuatro países no es una meta normativa.

### Visualización 3 — ¿Qué diferencias de perfil digital ocultan el DRS?

In [15]:
fig3 = fig_profiles(dim); fig3.show()
display(Markdown(f"""**¿Qué observo?** {describe_profiles(dim, drs)}

**¿Qué significa?** Dos economías con DRS parecido pueden tener perfiles muy distintos (p. ej., mucha infraestructura y poca innovación, o lo contrario). El radar separa las cinco dimensiones que el DRS promedia.

**¿Qué no puedo concluir?** Que un perfil más "grande" sea siempre mejor: las dimensiones tienen distinta importancia según el objetivo de política, y cada eje se construye con solo 1–2 indicadores."""))

**¿Qué observo?** México muestra un perfil que el DRS promedia: en «Actividad económica digital» ocupa el lugar 3 (frente al 5 del DRS global).

**¿Qué significa?** Dos economías con DRS parecido pueden tener perfiles muy distintos (p. ej., mucha infraestructura y poca innovación, o lo contrario). El radar separa las cinco dimensiones que el DRS promedia.

**¿Qué no puedo concluir?** Que un perfil más "grande" sea siempre mejor: las dimensiones tienen distinta importancia según el objetivo de política, y cada eje se construye con solo 1–2 indicadores.

### Visualización 4 (complementaria) — Valores observados y posición normalizada de cada indicador

In [16]:
fig4 = fig_heatmap(N, values); fig4.show()
display(Markdown("""**¿Qué observo?** Cada celda muestra el valor observado (texto) y su posición relativa (color: verde = mejor del grupo, rojo = peor; en I4 el color ya considera que menor costo es mejor).

**¿Qué significa?** Permite rastrear cada puntaje del DRS hasta el dato original y ver qué indicador empuja o frena a cada economía.

**¿Qué no puedo concluir?** Que los colores sean comparables entre columnas en términos absolutos: cada columna está normalizada de manera independiente."""))

**¿Qué observo?** Cada celda muestra el valor observado (texto) y su posición relativa (color: verde = mejor del grupo, rojo = peor; en I4 el color ya considera que menor costo es mejor).

**¿Qué significa?** Permite rastrear cada puntaje del DRS hasta el dato original y ver qué indicador empuja o frena a cada economía.

**¿Qué no puedo concluir?** Que los colores sean comparables entre columnas en términos absolutos: cada columna está normalizada de manera independiente.

## 7. Análisis complementario — Opción B: agrupamiento (clustering)
Con solo cinco observaciones, el clustering es **exploratorio**: sirve para describir perfiles, no para "descubrir" grupos estables. Se aplica K-means sobre los indicadores normalizados y estandarizados, y se elige *k* (2 o 3) por *silhouette*.

In [17]:
cl = cluster_profiles(N)
lab = cl["labels"]
print("Silhouette por k:", {k: round(v, 3) for k, v in cl["silhouettes"].items()}, "→ k =", cl["k"])
print("Indicadores usados (completos en las 5 economías):", cl["used"])
fig5 = fig_clusters(cl, drs); fig5.show()

rel = cl["centroids"] - N[cl["used"]].mean() * 100
partes = []
for g in sorted(lab.unique()):
    miembros = ", ".join(COUNTRIES[c] for c in lab[lab == g].index)
    fuertes = ", ".join(META[c]["short"] for c in rel.loc[g].nlargest(2).index)
    debiles = ", ".join(META[c]["short"] for c in rel.loc[g].nsmallest(2).index)
    partes.append(f"* **Grupo {g}** ({miembros}): relativamente alto en *{fuertes}*; relativamente bajo en *{debiles}*.")
orden = list(drs.sort_values(ascending=False).index)
cambios = sum(lab[a] != lab[b] for a, b in zip(orden, orden[1:]))
coincide = cambios == cl["k"] - 1
display(Markdown("\n".join(partes) + f"""

**Qué aporta frente al ranking del DRS:** {"los grupos coinciden con cortes contiguos del ranking, es decir, el agrupamiento confirma la separación del DRS pero además explica *en qué indicadores* se diferencian." if coincide else "los grupos NO son cortes contiguos del ranking: economías con DRS similar pertenecen a perfiles distintos, información que el ranking por sí solo no muestra."}

**Limitaciones:** n = 5; el resultado cambia si se agregan países o indicadores; K-means asume grupos compactos y usa distancia euclidiana."""))

Silhouette por k: {2: 0.287, 3: 0.198} → k = 2
Indicadores usados (completos en las 5 economías): ['fixed_broadband', 'secure_servers', 'internet_users', 'broadband_basket', 'ict_services_exports', 'ict_goods_exports', 'rd_expenditure', 'patents_per_million']


* **Grupo 1** (México): relativamente alto en *Exportaciones bienes TIC, Exportaciones servicios TIC*; relativamente bajo en *Gasto en I+D (% PIB), Banda ancha fija*.
* **Grupo 2** (Estados Unidos, Alemania, China, Japón): relativamente alto en *Gasto en I+D (% PIB), Banda ancha fija*; relativamente bajo en *Exportaciones bienes TIC, Exportaciones servicios TIC*.

**Qué aporta frente al ranking del DRS:** los grupos coinciden con cortes contiguos del ranking, es decir, el agrupamiento confirma la separación del DRS pero además explica *en qué indicadores* se diferencian.

**Limitaciones:** n = 5; el resultado cambia si se agregan países o indicadores; K-means asume grupos compactos y usa distancia euclidiana.

## 8. Brecha digital de México
Se agrupan los indicadores en los seis ámbitos solicitados. **No hay un indicador directo de *calidad del servicio*** (velocidad, latencia) en el conjunto: es una limitación que se declara en lugar de forzar un proxy.

In [18]:
ambitos = {"Disponibilidad (infraestructura)": ["fixed_broadband", "secure_servers"],
           "Acceso / uso": ["internet_users"],
           "Asequibilidad": ["broadband_basket"],
           "Calidad del servicio": [],
           "Capacidades (I+D e innovación)": ["rd_expenditure", "patents_per_million"],
           "Resultados (actividad económica digital)": ["ict_services_exports", "ict_goods_exports"]}
filas = []
for a, cols in ambitos.items():
    if not cols:
        filas.append(dict(ámbito=a, indicadores="— (sin indicador en el conjunto)", México=np.nan, prom_otros=np.nan, brecha=np.nan, lugar_MEX=""))
        continue
    s = N[cols].mean(axis=1) * 100
    filas.append(dict(ámbito=a, indicadores=", ".join(META[c]["id"] for c in cols), México=s["MEX"], prom_otros=s.drop("MEX").mean(),
                      brecha=s["MEX"] - s.drop("MEX").mean(), lugar_MEX=int(s.rank(ascending=False, method="min")["MEX"])))
tabla_brecha = pd.DataFrame(filas)
display(tabla_brecha)
principal = tabla_brecha.dropna(subset=["brecha"]).sort_values("brecha").iloc[0]
display(Markdown(f"""**Principal brecha de México:** «{principal['ámbito']}» ({principal['indicadores']}), con {principal['brecha']:+.0f} puntos frente al promedio de las otras cuatro economías (lugar {principal['lugar_MEX']} de 5). Detalle por indicador en la Visualización 2.

**¿Por qué una alta cobertura de red no implica alto uso o aprovechamiento?** La cobertura mide la *oferta* (disponibilidad de la señal), no la *demanda efectiva*. Para que la red se traduzca en aprovechamiento se necesitan: **asequibilidad** (precio del servicio y de los dispositivos frente al ingreso), **calidad** suficiente (velocidad y estabilidad) para usos productivos, **capacidades** (habilidades digitales, confianza), **contenidos y servicios útiles** en el idioma y contexto local, y **empresas** capaces de incorporar lo digital a sus procesos. En este conjunto, esa distinción se observa comparando I1–I2 (disponibilidad) con I3 (uso) e I4 (costo): México registra {values.loc['MEX','fixed_broadband']:.1f} suscripciones de banda ancha fija por 100 hab. y {values.loc['MEX','internet_users']:.1f}% de personas usuarias de Internet."""))

,ámbito,indicadores,México,prom_otros,brecha,lugar_MEX
0,Disponibilidad (infraestructura),"I1, I2",0.000,66.830,-66.830,5
1,Acceso / uso,I3,0.000,71.828,-71.828,5
2,Asequibilidad,I4,0.000,73.363,-73.363,5
3,Calidad del servicio,— (sin indicador en el conjunto),NaN,NaN,NaN,
4,Capacidades (I+D e innovación),"I7, I8",0.000,73.889,-73.889,5
5,Resultados (actividad económica digital),"I5, I6",23.554,39.996,-16.442,3


**Principal brecha de México:** «Capacidades (I+D e innovación)» (I7, I8), con -74 puntos frente al promedio de las otras cuatro economías (lugar 5 de 5). Detalle por indicador en la Visualización 2.

**¿Por qué una alta cobertura de red no implica alto uso o aprovechamiento?** La cobertura mide la *oferta* (disponibilidad de la señal), no la *demanda efectiva*. Para que la red se traduzca en aprovechamiento se necesitan: **asequibilidad** (precio del servicio y de los dispositivos frente al ingreso), **calidad** suficiente (velocidad y estabilidad) para usos productivos, **capacidades** (habilidades digitales, confianza), **contenidos y servicios útiles** en el idioma y contexto local, y **empresas** capaces de incorporar lo digital a sus procesos. En este conjunto, esa distinción se observa comparando I1–I2 (disponibilidad) con I3 (uso) e I4 (costo): México registra 20.8 suscripciones de banda ancha fija por 100 hab. y 83.1% de personas usuarias de Internet.

## 9. Preparación para IA — marco de las 4C

In [19]:
c4 = pd.DataFrame([{"Indicador": f'{i["id"]} {i["short"]}', "Dimensión 4C": i["c4"]} for i in INDICATORS])
display(c4)
display(Markdown("""
| Dimensión | Qué cubre nuestro conjunto | Qué falta |
|---|---|---|
| **Connectivity** | I1 (banda ancha fija), I3 (uso), I4 (asequibilidad) | Velocidad/latencia, cobertura 4G/5G, brecha urbano-rural |
| **Compute** | Solo proxies débiles: I2 (servidores seguros) e I6 (bienes TIC exportados) | Capacidad instalada de centros de datos y nube, acceso a GPU/HPC, energía para cómputo |
| **Context** | Prácticamente nada (I5 solo de forma indirecta) | Datos abiertos y de calidad, corpus en español y lenguas locales, aplicaciones y adopción empresarial, regulación de datos |
| **Competency** | I7 (gasto en I+D) e I8 (patentes) como capacidad general de innovación | Talento en IA/STEM, investigadores en IA, publicaciones y adopción de IA en empresas |

**¿Son suficientes los datos para afirmar que algún país está preparado para desarrollar y usar sistemas avanzados de IA?** **No.** Solo cubrimos bien *Connectivity* y de forma parcial *Competency*; *Compute* y *Context* están casi sin medir. Un DRS alto indica buena base digital general, no capacidad para IA avanzada. Para sostener esa afirmación haría falta información adicional: capacidad de cómputo y centros de datos (incluidos supercómputo/GPU), disponibilidad y calidad de datos y corpus por idioma, talento especializado (graduados STEM, investigadores en IA), inversión pública y privada en IA, adopción de IA por empresas, marco regulatorio y suministro energético.
"""))

,Indicador,Dimensión 4C
0,I1 Banda ancha fija,Connectivity
1,I2 Servidores seguros,Compute (proxy)
2,I3 Uso de Internet,Connectivity
3,I4 Costo banda ancha (% INB pc),Connectivity
4,I5 Exportaciones servicios TIC,Context/Competency (indirecto)
5,I6 Exportaciones bienes TIC,Compute (indirecto)
6,I7 Gasto en I+D (% PIB),Competency
7,I8 Patentes residentes /millón,Competency



| Dimensión | Qué cubre nuestro conjunto | Qué falta |
|---|---|---|
| **Connectivity** | I1 (banda ancha fija), I3 (uso), I4 (asequibilidad) | Velocidad/latencia, cobertura 4G/5G, brecha urbano-rural |
| **Compute** | Solo proxies débiles: I2 (servidores seguros) e I6 (bienes TIC exportados) | Capacidad instalada de centros de datos y nube, acceso a GPU/HPC, energía para cómputo |
| **Context** | Prácticamente nada (I5 solo de forma indirecta) | Datos abiertos y de calidad, corpus en español y lenguas locales, aplicaciones y adopción empresarial, regulación de datos |
| **Competency** | I7 (gasto en I+D) e I8 (patentes) como capacidad general de innovación | Talento en IA/STEM, investigadores en IA, publicaciones y adopción de IA en empresas |

**¿Son suficientes los datos para afirmar que algún país está preparado para desarrollar y usar sistemas avanzados de IA?** **No.** Solo cubrimos bien *Connectivity* y de forma parcial *Competency*; *Compute* y *Context* están casi sin medir. Un DRS alto indica buena base digital general, no capacidad para IA avanzada. Para sostener esa afirmación haría falta información adicional: capacidad de cómputo y centros de datos (incluidos supercómputo/GPU), disponibilidad y calidad de datos y corpus por idioma, talento especializado (graduados STEM, investigadores en IA), inversión pública y privada en IA, adopción de IA por empresas, marco regulatorio y suministro energético.


## 10. Diagnóstico final (≤ 250 palabras)

In [20]:
diagnostico = build_diagnosis(values, N, drs)
(PROC / "diagnosis.txt").write_text(diagnostico, encoding="utf-8")
res_out = pd.concat([N.add_prefix("norm_"), contrib.add_prefix("aporte_"), dim.add_prefix("dim_"), drs.rename("DRS")], axis=1)
res_out.insert(0, "country", [COUNTRIES[c] for c in res_out.index])
res_out.to_csv(PROC / "drs_results.csv", encoding="utf-8")
display(Markdown(diagnostico))
print("\nPalabras:", len(diagnostico.split()))

1) México obtiene un DRS de 4.7/100 y ocupa el lugar 5 de 5 (rango de las otras cuatro: 52.2–75.7). El índice es relativo a este grupo, no absoluto.

2) Principal fortaleza: Exportaciones bienes TIC — 12.45 (% de exportaciones de bienes), lugar 2 de 5 (+11 puntos frente al promedio de los otros cuatro).

3) Principal brecha: Gasto en I+D (% PIB) — 0.27 (% del PIB), lugar 5 de 5 (-91 puntos).

4) Japón es el punto de comparación más útil: tiene el perfil más cercano al de México (distancia euclidiana mínima en los indicadores normalizados) y la diferencia más grande con él está en «Patentes residentes /millón», por lo que es la brecha más accionable; el contraste máximo es con China.

5) Limitación principal: solo cinco economías y ocho indicadores; la normalización min-max depende de los extremos del grupo, varios indicadores son proxies (p. ej., servidores seguros como cómputo) y ninguno mide calidad del servicio, habilidades ni IA. Además, los años de observación pueden diferir entre indicadores (ver Source Log).


Palabras: 173


## 11. Entregables generados
`data/raw/` (respuestas originales) · `data/processed/digital_economy_clean.csv` · `source_log.csv` · `data_dictionary.csv` · `data/processed/drs_results.csv` y `diagnosis.txt` (usados por el dashboard) · `dashboard/app.py` (`streamlit run dashboard/app.py`).